# 002180 纳思达 — 极海微集成电路总营收提取

**目标**：从 GCS 年报 PDF 中提取每年：
- `mcu_revenue_yuan`：「芯片」行营业收入（MCU口径）
- `total_revenue_yuan`：「集成电路」产业总营收（极海微全部IC产品合计）

**原理**：先用 `gsutil` 把 PDF 下载到 Colab 本地，再上传到 Gemini Files API，
避免 GCS→BytesIO 流式传输的超时问题。

**运行前**：确保已在 Colab 中通过 `google.colab.auth.authenticate_user()` 完成 GCP 认证。

In [ ]:
# ── 1. 安装依赖 ──────────────────────────────────────────────────────────────
!pip install -q google-genai google-cloud-storage

In [ ]:
# ── 2. GCP 认证（Colab 内置） ─────────────────────────────────────────────────
from google.colab import auth
auth.authenticate_user()
print('✓ GCP 认证完成')

In [ ]:
# ── 3. 配置 ─────────────────────────────────────────────────────────────────
GCP_PROJECT  = 'st-china-ai-force'
GCS_BUCKET   = 'st-finance-reports'
SYMBOL       = '002180'
COMPANY_NAME = '纳思达'
GEMINI_MODEL = 'gemini-2.0-flash'   # 或 'gemini-1.5-pro'

# Gemini API Key（在 Colab Secrets 里设 GEMINI_API_KEY，或直接填）
import os
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY') or userdata.get('VITE_GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')

if not GEMINI_API_KEY:
    GEMINI_API_KEY = input('请输入 Gemini API Key: ').strip()

print(f'✓ Project: {GCP_PROJECT}  Bucket: {GCS_BUCKET}  Model: {GEMINI_MODEL}')

In [ ]:
# ── 4. 列出 GCS 中 002180 的所有年报 PDF ──────────────────────────────────────
from google.cloud import storage
import re

gcs    = storage.Client(project=GCP_PROJECT)
bucket = gcs.bucket(GCS_BUCKET)
blobs  = list(bucket.list_blobs(prefix='reports/'))

pdfs = []
for blob in blobs:
    if not blob.name.lower().endswith('.pdf'):
        continue
    parts = blob.name.split('/')
    if len(parts) < 3:
        continue
    folder = parts[1]   # e.g. '002180_纳思达'
    if not folder.startswith(SYMBOL):
        continue
    filename = parts[-1]
    yr_m = re.search(r'(20\d{2})', filename)
    if not yr_m:
        continue
    year = int(yr_m.group(1))
    # 只处理年报（排除季报/半年报）
    if re.search(r'[一二三四]季|半年|interim|quarterly|Q[1-4]', filename, re.I):
        continue
    pdfs.append({'year': year, 'blob': blob.name, 'size_mb': round(blob.size / 1e6, 1)})

pdfs.sort(key=lambda x: x['year'])
print(f'找到 {len(pdfs)} 份年报:')
for p in pdfs:
    print(f"  {p['year']}  {p['blob'].split('/')[-1]}  ({p['size_mb']} MB)")

In [ ]:
# ── 5. Gemini 提取函数 ────────────────────────────────────────────────────────
import json, time, pathlib, tempfile
from google import genai

SYSTEM_PROMPT = """你是一位专业的半导体行业财务分析师。
任务：从A股上市公司年报PDF中，精确提取MCU（微控制器）产品的营业收入数据。

输出要求（严格JSON格式）：
{
  "mcu_revenue_yuan": <float或null>,
  "total_revenue_yuan": <float或null>,
  "mcu_revenue_note": "<数据来源，如：分产品营业收入表第X页 芯片行>",
  "total_revenue_note": "<total_revenue_yuan来源描述>",
  "gross_margin_pct": <float或null>,
  "confidence": "high|medium|low",
  "reasoning": "<简要说明>",
  "source_text": "<原文关键片段，≤300字>"
}

针对纳思达/极海（002180）的特别说明：
- 年报在「主营业务分产品情况」或「营业收入构成」表中，通常有以下行：
    芯片、模组、打印耗材、集成电路（产业合计）等
- mcu_revenue_yuan = 「芯片」行的营业收入（元），这是极海微MCU口径
- total_revenue_yuan = 「集成电路」分行业合计（或芯片+模组+其他IC类行之和），
    即极海微半导体业务总营收（通常比「芯片」行大40-70%，2024年≈14亿元）
- 注意：不要把打印耗材计入 total_revenue_yuan（打印耗材是纳思达集团另一业务）
- 若找不到集成电路合计行，则 total_revenue_yuan = 芯片 + 模组（若有）
- 营收单位：元（年报有时写万元或亿元，统一换算为元）
- 数字含千分位逗号如 "821,011,684.45" 即 821011684.45 元"""


def extract_one(pdf_local_path: str, year: int, company: str) -> dict | None:
    """Upload PDF to Gemini Files API and extract revenue data."""
    client = genai.Client(api_key=GEMINI_API_KEY)

    print(f'  上传 PDF 到 Gemini Files API...')
    uploaded = client.files.upload(
        file=pdf_local_path,
        config={'mime_type': 'application/pdf'},
    )
    print(f'  上传完成，文件 URI: {uploaded.uri}')

    prompt = (
        f"{SYSTEM_PROMPT}\n\n"
        f"公司：{company}（{SYMBOL}）\n财年：{year}年\n\n"
        "请从上传的年报PDF中提取MCU产品营业收入数据，输出严格JSON格式。"
    )

    resp = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[uploaded, prompt],
    )
    raw = resp.text.strip()

    # 删除 Files API 临时文件（节省配额）
    try:
        client.files.delete(name=uploaded.name)
    except Exception:
        pass

    # 解析 JSON
    try:
        # 处理 ```json ... ``` 包裹
        m = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw, re.DOTALL)
        if m:
            return json.loads(m.group(1))
        return json.loads(raw[raw.find('{'):])
    except Exception as e:
        print(f'  ⚠ JSON 解析失败: {e}')
        print(f'  原始响应:\n{raw[:500]}')
        return None

print('✓ 提取函数就绪')

In [ ]:
# ── 6. 主循环：下载 PDF → Gemini 提取 ────────────────────────────────────────
import tempfile, pathlib

FX_RATES = {  # CNY/USD 年度均值（用于换算 M$）
    2018: 6.6174, 2019: 6.8985, 2020: 6.8976, 2021: 6.4515,
    2022: 6.7261, 2023: 7.0809, 2024: 7.1900, 2025: 7.2200,
}

results = {}   # year → extracted dict
workdir = pathlib.Path(tempfile.mkdtemp())

for item in pdfs:
    year     = item['year']
    blob_name = item['blob']
    filename  = blob_name.split('/')[-1]
    local_pdf = workdir / f"{SYMBOL}_{year}_{filename}"

    print(f'\n{'='*60}')
    print(f'[{year}] {filename}  ({item["size_mb"]} MB)')

    # ① 下载到 Colab 本地（gsutil / GCS SDK，Colab 在 Google 机房，速度快）
    print(f'  下载 GCS PDF → {local_pdf.name}...')
    !gsutil -q cp "gs://{GCS_BUCKET}/{blob_name}" "{local_pdf}"

    if not local_pdf.exists() or local_pdf.stat().st_size < 10_000:
        print(f'  ✗ 下载失败或文件过小，跳过')
        continue
    print(f'  ✓ 下载完成 ({local_pdf.stat().st_size/1e6:.1f} MB)')

    # ② Gemini 提取
    result = extract_one(str(local_pdf), year, COMPANY_NAME)
    local_pdf.unlink(missing_ok=True)   # 删除本地 PDF，节省空间

    if result is None:
        print(f'  ✗ 提取失败')
        continue

    fx = FX_RATES.get(year, 7.2)
    mcu_y  = result.get('mcu_revenue_yuan')
    tot_y  = result.get('total_revenue_yuan')
    mcu_m  = round(mcu_y / fx / 1e6, 2) if mcu_y else None
    tot_m  = round(tot_y / fx / 1e6, 2) if tot_y else None
    conf   = result.get('confidence', '?')

    print(f'  ✓ mcu_revenue:   ¥{mcu_y:>15,.0f}  ({mcu_m} M$)' if mcu_y else '  ✓ mcu_revenue: null')
    print(f'  ✓ total_revenue: ¥{tot_y:>15,.0f}  ({tot_m} M$)' if tot_y else '  ✓ total_revenue: null')
    print(f'  confidence: {conf}  |  {result.get("mcu_revenue_note","")}')

    results[year] = result
    time.sleep(2)   # 避免触发 Gemini API 速率限制

print(f'\n\n完成：共提取 {len(results)}/{len(pdfs)} 年')

In [ ]:
# ── 7. 生成 mcu_known_data.json 的 002180 条目 ───────────────────────────────
# 复制下面的输出，粘贴替换 mcu_known_data.json 中 "002180" 的值

FX_RATES = {2018:6.6174,2019:6.8985,2020:6.8976,2021:6.4515,
            2022:6.7261,2023:7.0809,2024:7.1900,2025:7.2200}

output = {}
for year, r in sorted(results.items()):
    mcu_y = r.get('mcu_revenue_yuan')
    tot_y = r.get('total_revenue_yuan')
    if mcu_y is None:
        continue
    entry = {
        'mcu_revenue_yuan': mcu_y,
        'data_type':        'reported' if r.get('confidence') == 'high' else 'estimated',
        'confidence':       r.get('confidence', 'medium'),
        'source':           r.get('mcu_revenue_note', f'Gemini {GEMINI_MODEL} 提取'),
    }
    if tot_y is not None:
        entry['total_revenue_yuan'] = tot_y
        entry['total_revenue_note'] = r.get('total_revenue_note', '集成电路产业总营收，Gemini提取')
    if r.get('gross_margin_pct') is not None:
        entry['mcu_gross_margin'] = r['gross_margin_pct'] / 100
    output[str(year)] = entry

print('─' * 60)
print('粘贴以下内容到 mcu_known_data.json 中 "002180" 的值：')
print('─' * 60)
print(json.dumps(output, ensure_ascii=False, indent=2))

In [ ]:
# ── 8. （可选）直接写入 mcu_known_data.json ─────────────────────────────────
# 如果 Colab 已 mount 了项目目录，可直接写入；否则用上方输出手动复制

KNOWN_DATA_PATH = '/content/mcu-regional-competitor-dashboard/mcu_known_data.json'
# 如果是 Drive mount 改为:
# KNOWN_DATA_PATH = '/content/drive/MyDrive/mcu-dash/mcu_known_data.json'

import pathlib
p = pathlib.Path(KNOWN_DATA_PATH)
if p.exists():
    data = json.loads(p.read_text())
    data['002180'] = output
    p.write_text(json.dumps(data, ensure_ascii=False, indent=2))
    print(f'✓ 已写入 {KNOWN_DATA_PATH}')
    print(f'  002180 条目: {len(output)} 年')
else:
    print(f'⚠ 文件不存在: {KNOWN_DATA_PATH}')
    print('请手动将上方 JSON 粘贴到 mcu_known_data.json')

In [ ]:
# ── 9. 后续步骤（在 Cloud Shell 执行） ───────────────────────────────────────
print("""
后续步骤（在 Cloud Shell 或本地终端）：

  # 更新 mcu_known_data.json 后，重新生成 data.json
  python fetch_mcu_data.py

  # 验证
  python validate_data.py   # 必须显示 PASS

  # 提交
  git add data.json mcu_known_data.json
  git commit -m "feat(data): 002180 纳思达 total_revenue 2018-2025 from 年报提取"
  git push origin claude/code-review-hlYFP
""")